# Stat 220, Unit 4 Homework: Prediction and Choosing Predictors

`moving_jobs.csv` is 600 completed jobs at a moving company. `hours` is the crew's
time on site. The predictors that make sense are `volume_cuft`, `crew_size`, `stairs_flights`,
`miles`, and `packing_service`. The company also records `est_boxes`, `dispatcher_rating`,
and `weekend`.

```python
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split

jobs = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/moving_jobs.csv")
folds = KFold(5, shuffle=True, random_state=0)
jobs.head()
```

**Problem 1.** *Quoting a customer.* A customer is moving 650 cubic feet, with a crew of 2,
one flight of stairs, 20 miles, and no packing service.

Part a. Report the predicted hours, the confidence interval, and the prediction interval. Say in one sentence what each of the two intervals is about.

In [ ]:
fit = smf.ols("hours ~ volume_cuft + crew_size + stairs_flights + miles + packing_service",
              data=jobs).fit()

new = pd.DataFrame({"volume_cuft": [650], "crew_size": [2], "stairs_flights": [1],
                    "miles": [20], "packing_service": [0]})
print(fit.get_prediction(new).summary_frame(alpha=0.05).round(2))

_Your answer:_



Part b. The dispatcher has to block time on Tuesday's schedule for this one job. Which of the two intervals should they use, and why?

_Your answer:_



Part c. The owner asks a different question: across the next hundred jobs like this, what will our average time be? Which interval answers that one?

_Your answer:_



**Problem 2.** *A job that looks ordinary.* A customer reports 500 cubic feet and 90 boxes.

Part a. Each value is common on its own. Report how many completed jobs are near each value, and how many are near both.

In [ ]:
near_volume = jobs.volume_cuft.between(400, 600)
near_boxes = jobs.est_boxes.between(82, 98)

print("jobs near this volume    :", near_volume.sum())
print("jobs near this box count :", near_boxes.sum())
print("jobs near both           :", (near_volume & near_boxes).sum())

_Your answer:_



Part b. In three or four sentences, explain what your part a numbers mean for a prediction for this customer, and what you would tell the person who asked for it.

_Your answer:_



**Problem 3.** *How wrong will it be?* The owner wants one number for how far off a quote
typically is.

Part a. Report both numbers, and say which one you would give the owner.

In [ ]:
X = jobs[["volume_cuft", "crew_size", "stairs_flights", "miles", "packing_service"]]
y = jobs["hours"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(X_train, y_train)

print("on the rows it was fitted on:", round(np.sqrt(((y_train - m.predict(X_train))**2).mean()), 3))
print("on the rows held back       :", round(np.sqrt(((y_test - m.predict(X_test))**2).mean()), 3))

_Your answer:_



Part b. Explain in two or three sentences why those two numbers differ, and why the gap would be larger for a model with many more predictors.

_Your answer:_



**Problem 4.** *Which predictors belong?* The branch office has 60 completed jobs so far.

Part a. Report the cross-validated error and the AIC for each set. Say which set you would ship.

In [ ]:
branch = jobs.sample(60, random_state=5)
sets = {
    "volume only": ["volume_cuft"],
    "the five that make sense": ["volume_cuft", "crew_size", "stairs_flights",
                                 "miles", "packing_service"],
    "those five plus three junk": ["volume_cuft", "crew_size", "stairs_flights",
                                   "miles", "packing_service",
                                   "est_boxes", "dispatcher_rating", "weekend"],
}
for name, cols in sets.items():
    mse = -cross_val_score(LinearRegression(), branch[cols], branch.hours, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    aic = smf.ols("hours ~ " + " + ".join(cols), data=branch).fit()
    print(f"{name:<28} CV error {np.sqrt(mse):.3f}   R2 {aic.rsquared:.4f}   AIC {aic.aic:.1f}")

_Your answer:_



Part b. One set has the highest $R^2$ and is still not the set you would ship. Explain what $R^2$ is doing here in two or three sentences.

_Your answer:_



Part c. A colleague says to run the same comparison on all 600 jobs instead of 60, and expects the junk columns to look better there. Would you expect the same conclusion? Say why.

_Your answer:_



**Problem 5.** *What can and cannot be said.* No computer.

Part a. A colleague tried 65 columns, kept the 11 with p-values under 0.05, and reports $R^2 = 0.96$ with every predictor significant. Write the two or three sentences you would say in response.

_Your answer:_



Part b. A lasso fit sends `weekend` to exactly zero. Does that show weekends have no effect on how long a move takes? Explain.

_Your answer:_



Part c. Your model predicts well and `est_boxes` has a large coefficient. The owner asks whether telling customers to use fewer, larger boxes would shorten jobs. Answer in two or three sentences.

_Your answer:_

